# CR-LT Support Router Training

This notebook is for the CR-LT-only router stage. It uses support-alignment supervision from the CR-LT annotations, reads raw data from `Router_for_FeDeLiS/raw_data/crlt`, preprocesses into `Router_for_FeDeLiS/preprocessed/crlt`, and trains with CR-LT-specific support metrics.


In [28]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print({'is_colab': IS_COLAB, 'python': sys.executable})

# Choose one: 'colab_git', 'colab_drive', 'local_repo'
BOOTSTRAP_MODE = 'colab_drive'

# Fill this if using BOOTSTRAP_MODE='colab_git'
GIT_REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
GIT_BRANCH = 'main'
COLAB_CLONE_DIR = Path('/content/project')

# Fill this if using BOOTSTRAP_MODE='colab_drive'
DRIVE_PROJECT_PATH = Path('/content/drive/MyDrive/MIT_nlp/project')
COLAB_DRIVE_COPY_DIR = Path('/content/project')

# Use this if BOOTSTRAP_MODE='local_repo'
LOCAL_PROJECT_ROOT = Path('/Users/tzhao/Desktop/Harvard/MIT_nlp/project')

# Run identity and training hyperparameters
RUN_NAME = 'default'
MODEL_NAME = 'distilbert-base-uncased'
FREEZE_MODE = 'last1'          # frozen | last1 | full
MAX_LENGTH = 96
POINTWISE_BATCH_SIZE = 32
PAIRWISE_BATCH_SIZE = 32
EPOCHS = 30
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
N_SPLITS = 5
PATIENCE = 5
SEED = 42
POINTWISE_LOSS_WEIGHT = 1.0
PAIRWISE_LOSS_WEIGHT = 0.5
REWARD_COST_WEIGHT = 0.05
PAIRWISE_MARGIN_WEIGHT = 1.0
REBUILD_PREPROCESSED = False


{'is_colab': True, 'python': '/usr/bin/python3'}


In [29]:
def run(cmd, cwd=None, check=True):
    if isinstance(cmd, list):
        printable = ' '.join(str(x) for x in cmd)
    else:
        printable = cmd
    print(f'\n$ {printable}')
    return subprocess.run(cmd, cwd=cwd, check=check)


def ensure_python_packages():
    packages = [
        'torch',
        'transformers',
        'scikit-learn',
        'pandas',
        'numpy',
    ]
    run([sys.executable, '-m', 'pip', 'install', '-q', *packages])


def bootstrap_project_root() -> Path:
    if BOOTSTRAP_MODE == 'local_repo':
        project_root = LOCAL_PROJECT_ROOT
        if not project_root.exists():
            raise FileNotFoundError(f'LOCAL_PROJECT_ROOT not found: {project_root}')
        return project_root

    if not IS_COLAB:
        raise RuntimeError(
            'BOOTSTRAP_MODE is set for Colab, but this notebook is not running in Colab. ' 
            'Use local_repo or switch to a Colab runtime.'
        )

    if BOOTSTRAP_MODE == 'colab_git':
        if '<your-user>' in GIT_REPO_URL or '<your-repo>' in GIT_REPO_URL:
            raise ValueError('Set GIT_REPO_URL before running the bootstrap cell.')
        if COLAB_CLONE_DIR.exists() and any(COLAB_CLONE_DIR.iterdir()):
            print(f'Using existing clone at {COLAB_CLONE_DIR}')
        else:
            run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, str(COLAB_CLONE_DIR)])
        return COLAB_CLONE_DIR

    if BOOTSTRAP_MODE == 'colab_drive':
        from google.colab import drive
        drive.mount('/content/drive')
        if not DRIVE_PROJECT_PATH.exists():
            raise FileNotFoundError(f'DRIVE_PROJECT_PATH not found: {DRIVE_PROJECT_PATH}')
        if COLAB_DRIVE_COPY_DIR.exists():
            shutil.rmtree(COLAB_DRIVE_COPY_DIR)
        shutil.copytree(DRIVE_PROJECT_PATH, COLAB_DRIVE_COPY_DIR)
        return COLAB_DRIVE_COPY_DIR

    raise ValueError(f'Unknown BOOTSTRAP_MODE: {BOOTSTRAP_MODE}')


ensure_python_packages()
PROJECT_ROOT = bootstrap_project_root().resolve()
ROUTER_DIR = PROJECT_ROOT / 'Router_for_FeDeLiS'
TRAINER_PATH = ROUTER_DIR / 'scripts' / 'crlt' / 'train_bert_crlt_support_ranker.py'
PREPROCESS_PATH = ROUTER_DIR / 'scripts' / 'crlt' / 'preprocess_crlt_support_training_data.py'
RAW_DATA_DIR = ROUTER_DIR / 'raw_data' / 'crlt'
PREPROCESSED_DIR = ROUTER_DIR / 'preprocessed' / 'crlt'
OUTPUT_DIR = ROUTER_DIR / 'outputs' / 'crlt' / f'bert_crlt_support_ranker_{RUN_NAME}'

print({
    'project_root': str(PROJECT_ROOT),
    'router_dir': str(ROUTER_DIR),
    'preprocess_exists': PREPROCESS_PATH.exists(),
    'trainer_exists': TRAINER_PATH.exists(),
    'raw_data_dir': str(RAW_DATA_DIR),
    'preprocessed_dir': str(PREPROCESSED_DIR),
})



$ /usr/bin/python3 -m pip install -q torch transformers scikit-learn pandas numpy
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'project_root': '/content/project', 'router_dir': '/content/project/Router_for_FeDeLiS', 'preprocess_exists': True, 'trainer_exists': True, 'raw_data_dir': '/content/project/Router_for_FeDeLiS/raw_data/crlt', 'preprocessed_dir': '/content/project/Router_for_FeDeLiS/preprocessed/crlt'}


In [30]:
required_files = [
    PREPROCESSED_DIR / 'router_query_table.jsonl',
    PREPROCESSED_DIR / 'router_action_table.jsonl',
    PREPROCESSED_DIR / 'router_pairwise_table.jsonl',
]

need_preprocess = REBUILD_PREPROCESSED or not all(path.exists() for path in required_files)
print({'need_preprocess': need_preprocess})

if need_preprocess:
    run([sys.executable, str(PREPROCESS_PATH)], cwd=PROJECT_ROOT)

for path in required_files:
    print(path, path.exists(), path.stat().st_size if path.exists() else None)


{'need_preprocess': False}
/content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl True 146606
/content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl True 1526550
/content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl True 2928989


In [31]:
import json
import pandas as pd

query_path = PREPROCESSED_DIR / 'router_query_table.jsonl'
action_path = PREPROCESSED_DIR / 'router_action_table.jsonl'
pairwise_path = PREPROCESSED_DIR / 'router_pairwise_table.jsonl'

for path in [query_path, action_path, pairwise_path]:
    with path.open('r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    print(path.name, count)

query_rows = [json.loads(line) for line in query_path.open('r', encoding='utf-8') if line.strip()]
action_rows = [json.loads(line) for line in action_path.open('r', encoding='utf-8') if line.strip()]

summary = {
    'queries': len(query_rows),
    'actions': len(action_rows),
    'queries_with_support_label': sum(row['support_label_depth'] is not None for row in query_rows),
    'queries_needing_more_than_baseline': sum(bool(row['gate_target_need_better_than_baseline']) for row in query_rows),
    'baseline_best_by_support': sum(bool(row['baseline_is_best_by_support_cost']) for row in query_rows),
}
print(summary)
display(pd.DataFrame([summary]))
display(pd.DataFrame(query_rows).head(5))


router_query_table.jsonl 101
router_action_table.jsonl 735
router_pairwise_table.jsonl 2680
{'queries': 101, 'actions': 735, 'queries_with_support_label': 50, 'queries_needing_more_than_baseline': 19, 'baseline_best_by_support': 82}


,queries,actions,queries_with_support_label,queries_needing_more_than_baseline,baseline_best_by_support
0,101,735,50,19,82


,query_id,benchmark,task_type,split,status,query,q_entities,q_entity_count,ground_truth,hop,...,best_runtime_sec,best_llm_f1,best_support_precision,best_support_recall,best_support_fbeta,best_support_minus_baseline,baseline_is_best_by_support_cost,baseline_is_within_best_epsilon,gate_target_need_better_than_baseline,has_positive_support_label
0,S167,CL-LT-KGQA,qa,train,resolved,Would Marjan Faritous look odd with a moustache?,[Marjan Faritous],1,[True],2,...,6.501,1.0,1.00,1.0,1.000000,0.000000,True,True,False,True
1,S174,CL-LT-KGQA,qa,train,unresolved,Did Alan Hamel's wife pass away?,[Alan Hamel],1,[True],2,...,6.029,0.0,1.00,0.5,0.555556,0.000000,True,True,False,True
2,S46,CL-LT-KGQA,qa,train,resolved,Could Quidam de Revel fit in HMS Abercrombie?,"[Quidam de Revel, HMS Abercrombie]",2,[True],3,...,10.528,0.0,0.25,0.5,0.416667,0.416667,False,False,True,True
3,S3,CL-LT-KGQA,qa,train,resolved,Does Leopold Lanner's wife make a living using...,[Leopold Lanner],1,[True],2,...,5.879,1.0,1.00,0.5,0.555556,0.000000,True,True,False,True
4,S159,CL-LT-KGQA,qa,train,resolved,is Marjan Faritous likely to be familiar with ...,"[Marjan Faritous, Elijah]",2,[True],3,...,10.957,0.0,0.00,0.0,0.000000,0.000000,True,True,False,False


## CR-LT Sweep Runner

This notebook supports two CR-LT training modes:
- `weighted_logistic`: the original pairwise ranking objective
- `dynamic_margin`: the CR-LT margin-aware pairwise ranking objective

Both are evaluated with CR-LT support-faithfulness metrics such as `mean_pred_support_fbeta`.


In [ ]:
import pandas as pd
import time

DRIVE_SWEEP_ROOT = Path('/content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/crlt/router_sweeps')
RESUME_IF_DRIVE_RESULT_EXISTS = True
OVERWRITE_DRIVE_RUN_DIR = True

MODEL_NAME = 'distilbert-base-uncased'
FREEZE_MODE = 'last1'
MAX_LENGTH = 96
POINTWISE_BATCH_SIZE = 32
PAIRWISE_BATCH_SIZE = 32
EPOCHS = 12
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
N_SPLITS = 3
PATIENCE = 3
SEED = 42

# CR-LT defaults
DEFAULT_PAIRWISE_LOSS_MODE = 'weighted_logistic'
ALTERNATIVE_PAIRWISE_LOSS_MODE = 'dynamic_margin'
DEFAULT_POINTWISE_LOSS_WEIGHT = 1.0
DEFAULT_PAIRWISE_LOSS_WEIGHT = 2.0
REWARD_COST_WEIGHT = 0.05
PAIRWISE_MARGIN_WEIGHT = 1.0
DEFAULT_DYNAMIC_MARGIN_SCALE = 0.5

DEFAULT_PAIRWISE_LOSS_WEIGHTS = [2.0, 3.0, 4.0]
DEFAULT_POINTWISE_LOSS_WEIGHTS = [0.5, 1.0, 2.0, 4.0]

def build_train_cmd(run_output_dir: Path, cfg: dict) -> list[str]:
    cmd = [
        sys.executable,
        str(TRAINER_PATH),
        '--query-path', str(PREPROCESSED_DIR / 'router_query_table.jsonl'),
        '--action-path', str(PREPROCESSED_DIR / 'router_action_table.jsonl'),
        '--pairwise-path', str(PREPROCESSED_DIR / 'router_pairwise_table.jsonl'),
        '--output-dir', str(run_output_dir),
        '--model-name', MODEL_NAME,
        '--freeze-mode', FREEZE_MODE,
        '--max-length', str(MAX_LENGTH),
        '--pointwise-batch-size', str(POINTWISE_BATCH_SIZE),
        '--pairwise-batch-size', str(PAIRWISE_BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--encoder-learning-rate', str(ENCODER_LR),
        '--head-learning-rate', str(HEAD_LR),
        '--weight-decay', str(WEIGHT_DECAY),
        '--dropout', str(DROPOUT),
        '--n-splits', str(N_SPLITS),
        '--early-stopping-patience', str(PATIENCE),
        '--seed', str(SEED),
        '--pointwise-loss-weight', str(cfg['pointwise_loss_weight']),
        '--pairwise-loss-weight', str(cfg['pairwise_loss_weight']),
        '--reward-cost-weight', str(cfg['reward_cost_weight']),
        '--pairwise-margin-weight', str(PAIRWISE_MARGIN_WEIGHT),
        '--pairwise-loss-mode', cfg['pairwise_loss_mode'],
    ]
    if cfg['pairwise_loss_mode'] == 'dynamic_margin':
        cmd.extend(['--pairwise-dynamic-margin-scale', str(cfg['pairwise_dynamic_margin_scale'])])
    return cmd

def load_summary_wide(summary_csv_path: Path) -> dict:
    summary_df = pd.read_csv(summary_csv_path)
    row = {rec['metric']: rec['mean'] for rec in summary_df.to_dict('records')}
    std_row = {f"{rec['metric']}__std": rec['std'] for rec in summary_df.to_dict('records')}
    row.update(std_row)
    return row

def copy_run_to_drive(local_output_dir: Path, drive_run_dir: Path) -> None:
    if drive_run_dir.exists() and OVERWRITE_DRIVE_RUN_DIR:
        shutil.rmtree(drive_run_dir)
    drive_run_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_run_dir)

def write_drive_aggregate(rows: list[dict], drive_group_dir: Path) -> None:
    aggregate_df = pd.DataFrame(rows)
    aggregate_df = aggregate_df.sort_values(['mean_pred_support_fbeta', 'exact_accuracy'], ascending=[False, False])
    aggregate_csv = drive_group_dir / 'sweep_summary.csv'
    aggregate_jsonl = drive_group_dir / 'sweep_summary.jsonl'
    aggregate_df.to_csv(aggregate_csv, index=False)
    with aggregate_jsonl.open('w', encoding='utf-8') as handle:
        for rec in aggregate_df.to_dict('records'):
            handle.write(json.dumps(rec, ensure_ascii=False) + '\n')

def build_mode_configs(*, mode_name: str, sweep_group_name: str, pairwise_loss_weights: list[float], reward_cost_weight: float, pointwise_loss_weight: float = DEFAULT_POINTWISE_LOSS_WEIGHT, dynamic_margin_scales: list[float] | None = None) -> list[dict]:
    configs = []
    if mode_name == 'dynamic_margin':
        dynamic_margin_scales = dynamic_margin_scales or [DEFAULT_DYNAMIC_MARGIN_SCALE]
        for margin_scale in dynamic_margin_scales:
            for pairwise_loss_weight in pairwise_loss_weights:
                run_name = (
                    f'{mode_name}_'
                    f'pair{str(pairwise_loss_weight).replace(".", "p")}_'
                    f'point{str(pointwise_loss_weight).replace(".", "p")}_'
                    f'cost{str(reward_cost_weight).replace(".", "p")}_'
                    f'mscale{str(margin_scale).replace(".", "p")}_'
                    f'{FREEZE_MODE}'
                )
                configs.append({
                    'sweep_group': sweep_group_name,
                    'run_name': run_name,
                    'pairwise_loss_mode': mode_name,
                    'pairwise_dynamic_margin_scale': margin_scale,
                    'pointwise_loss_weight': pointwise_loss_weight,
                    'pairwise_loss_weight': pairwise_loss_weight,
                    'reward_cost_weight': reward_cost_weight,
                })
    else:
        for pairwise_loss_weight in pairwise_loss_weights:
            run_name = (
                f'{mode_name}_'
                f'pair{str(pairwise_loss_weight).replace(".", "p")}_'
                f'point{str(pointwise_loss_weight).replace(".", "p")}_'
                f'cost{str(reward_cost_weight).replace(".", "p")}_'
                f'{FREEZE_MODE}'
            )
            configs.append({
                'sweep_group': sweep_group_name,
                'run_name': run_name,
                'pairwise_loss_mode': mode_name,
                'pairwise_dynamic_margin_scale': None,
                'pointwise_loss_weight': pointwise_loss_weight,
                'pairwise_loss_weight': pairwise_loss_weight,
                'reward_cost_weight': reward_cost_weight,
            })
    return configs

def build_objective_weight_configs(*, pairwise_loss_mode: str, sweep_group_name: str, pointwise_loss_weights: list[float], reward_cost_weight: float, pairwise_loss_weight: float = DEFAULT_PAIRWISE_LOSS_WEIGHT, dynamic_margin_scale: float = DEFAULT_DYNAMIC_MARGIN_SCALE) -> list[dict]:
    configs = []
    for pointwise_loss_weight in pointwise_loss_weights:
        run_name = (
            f'objective_'
            f'{pairwise_loss_mode}_'
            f'point{str(pointwise_loss_weight).replace(".", "p")}_'
            f'pair{str(pairwise_loss_weight).replace(".", "p")}_'
            f'cost{str(reward_cost_weight).replace(".", "p")}_'
            f'{FREEZE_MODE}'
        )
        configs.append({
            'sweep_group': sweep_group_name,
            'run_name': run_name,
            'pairwise_loss_mode': pairwise_loss_mode,
            'pairwise_dynamic_margin_scale': dynamic_margin_scale if pairwise_loss_mode == 'dynamic_margin' else None,
            'pointwise_loss_weight': pointwise_loss_weight,
            'pairwise_loss_weight': pairwise_loss_weight,
            'reward_cost_weight': reward_cost_weight,
        })
    return configs

def run_one_config(cfg: dict, drive_group_dir: Path) -> dict:
    local_output_dir = ROUTER_DIR / 'outputs' / 'crlt' / f"bert_crlt_support_ranker_{cfg['sweep_group']}_{cfg['run_name']}"
    drive_run_dir = drive_group_dir / local_output_dir.name
    drive_summary_path = drive_run_dir / 'bert_crlt_support_ranker_cv_summary.csv'
    if RESUME_IF_DRIVE_RESULT_EXISTS and drive_summary_path.exists():
        print(f"Skipping existing Drive result: {cfg['run_name']}")
        wide = load_summary_wide(drive_summary_path)
        result = {
            'sweep_group': cfg['sweep_group'],
            'run_name': cfg['run_name'],
            'status': 'skipped_existing',
            'pairwise_loss_mode': cfg['pairwise_loss_mode'],
            'pairwise_dynamic_margin_scale': cfg['pairwise_dynamic_margin_scale'],
            'pointwise_loss_weight': cfg['pointwise_loss_weight'],
            'pairwise_loss_weight': cfg['pairwise_loss_weight'],
            'reward_cost_weight': cfg['reward_cost_weight'],
            'n_splits': N_SPLITS,
        }
        result.update(wide)
        return result
    if local_output_dir.exists():
        shutil.rmtree(local_output_dir)
    local_output_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_train_cmd(local_output_dir, cfg)
    start_time = time.time()
    run(cmd, cwd=PROJECT_ROOT)
    elapsed_sec = time.time() - start_time
    summary_csv = local_output_dir / 'bert_crlt_support_ranker_cv_summary.csv'
    if not summary_csv.exists():
        raise FileNotFoundError(f'Missing summary CSV for {cfg['run_name']}: {summary_csv}')
    copy_run_to_drive(local_output_dir, drive_run_dir)
    wide = load_summary_wide(summary_csv)
    result = {
        'sweep_group': cfg['sweep_group'],
        'run_name': cfg['run_name'],
        'status': 'completed',
        'pairwise_loss_mode': cfg['pairwise_loss_mode'],
        'pairwise_dynamic_margin_scale': cfg['pairwise_dynamic_margin_scale'],
        'pointwise_loss_weight': cfg['pointwise_loss_weight'],
        'pairwise_loss_weight': cfg['pairwise_loss_weight'],
        'reward_cost_weight': cfg['reward_cost_weight'],
        'n_splits': N_SPLITS,
        'elapsed_sec': elapsed_sec,
    }
    result.update(wide)
    metadata = {
        'project_root': str(PROJECT_ROOT),
        'trainer_path': str(TRAINER_PATH),
        'preprocessed_dir': str(PREPROCESSED_DIR),
        'output_dir': str(local_output_dir),
        'drive_run_dir': str(drive_run_dir),
        'command': cmd,
        'config': result,
    }
    with (drive_run_dir / 'run_metadata.json').open('w', encoding='utf-8') as handle:
        json.dump(metadata, handle, indent=2)
    print(f"Finished {cfg['run_name']} in {elapsed_sec / 60.0:.2f} minutes")
    return result

def execute_sweep(sweep_configs: list[dict]) -> tuple[Path, list[dict]]:
    if not sweep_configs:
        raise ValueError('sweep_configs is empty')
    sweep_group_name = sweep_configs[0]['sweep_group']
    drive_group_dir = DRIVE_SWEEP_ROOT / sweep_group_name
    drive_group_dir.mkdir(parents=True, exist_ok=True)
    sweep_results = []
    for cfg in sweep_configs:
        result = run_one_config(cfg, drive_group_dir)
        sweep_results.append(result)
        write_drive_aggregate(sweep_results, drive_group_dir)
        display(
            pd.DataFrame(sweep_results)
            .sort_values(['mean_pred_support_fbeta', 'exact_accuracy'], ascending=[False, False])
            .reset_index(drop=True)
        )
    print(f'Sweep outputs saved under: {drive_group_dir}')
    return drive_group_dir, sweep_results

def show_leaderboard(drive_group_dir: Path):
    aggregate_csv = drive_group_dir / 'sweep_summary.csv'
    aggregate_df = pd.read_csv(aggregate_csv)
    ranking_columns = [
        'run_name',
        'pairwise_loss_mode',
        'pairwise_dynamic_margin_scale',
        'pointwise_loss_weight',
        'pairwise_loss_weight',
        'reward_cost_weight',
        'mean_pred_support_fbeta',
        'mean_best_minus_pred_support_fbeta',
        'mean_pred_minus_baseline_support_fbeta',
        'exact_accuracy',
        'strategy_macro_f1',
        'depth_accuracy',
        'width_accuracy',
        'elapsed_sec',
        'status',
    ]
    available_columns = [col for col in ranking_columns if col in aggregate_df.columns]
    aggregate_df = aggregate_df[available_columns].sort_values(
        ['mean_pred_support_fbeta', 'exact_accuracy'],
        ascending=[False, False],
    ).reset_index(drop=True)
    print(f'CR-LT leaderboard for {drive_group_dir.name}')
    display(aggregate_df)


## Section 1: Default CR-LT Mode (`weighted_logistic`)

This is the default CR-LT mode. It uses the original pairwise ranking objective and sweeps `pairwise_loss_weight` under support-faithfulness evaluation.


In [33]:
DEFAULT_PAIRWISE_LOSS_WEIGHTS = [0.5, 1.0, 2.0]
DEFAULT_POINTWISE_LOSS_WEIGHTS = [1.0]

DEFAULT_MODE_SWEEP_GROUP_NAME = 'pair_weight_weighted_logistic_v1'
default_mode_configs = build_mode_configs(
    mode_name=DEFAULT_PAIRWISE_LOSS_MODE,
    sweep_group_name=DEFAULT_MODE_SWEEP_GROUP_NAME,
    pairwise_loss_weights=DEFAULT_PAIRWISE_LOSS_WEIGHTS,
    reward_cost_weight=REWARD_COST_WEIGHT,
)
display(pd.DataFrame(default_mode_configs))


,sweep_group,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair0p5_point1p0_cost0p05_last1,weighted_logistic,None,1.0,0.5,0.05
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair1p0_point1p0_cost0p05_last1,weighted_logistic,None,1.0,1.0,0.05
2,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_point1p0_cost0p05_last1,weighted_logistic,None,1.0,2.0,0.05


In [34]:
default_mode_drive_group_dir, default_mode_results = execute_sweep(default_mode_configs)



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair0p5_point1p0_cost0p05_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 0.5 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finished weighte

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair0p5_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,0.5,0.05,3,356.857891,...,0.073805,0.214397,0.087292,0.035607,0.01462,0.024662,0.037733,0.030925,0.127112,0.052282



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair1p0_point1p0_cost0p05_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 1.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finished weighte

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair1p0_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,1.0,0.05,3,358.625983,...,0.245499,0.155549,0.135883,0.033511,0.01462,0.024662,0.035052,0.028099,0.097455,0.052282
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair0p5_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,0.5,0.05,3,356.857891,...,0.073805,0.214397,0.087292,0.035607,0.01462,0.024662,0.037733,0.030925,0.127112,0.052282



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_pair_weight_weighted_logistic_v1_weighted_logistic_pair2p0_point1p0_cost0p05_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode weighted_logistic
Finished weighte

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,pair_weight_weighted_logistic_v1,weighted_logistic_pair1p0_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,1.0,0.05,3,358.625983,...,0.245499,0.155549,0.135883,0.033511,0.01462,0.024662,0.035052,0.028099,0.097455,0.052282
1,pair_weight_weighted_logistic_v1,weighted_logistic_pair2p0_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,2.0,0.05,3,337.766825,...,0.110069,0.099791,0.345632,0.032586,0.01462,0.024662,0.032770,0.024975,0.113294,0.052282
2,pair_weight_weighted_logistic_v1,weighted_logistic_pair0p5_point1p0_cost0p05_last1,completed,weighted_logistic,None,1.0,0.5,0.05,3,356.857891,...,0.073805,0.214397,0.087292,0.035607,0.01462,0.024662,0.037733,0.030925,0.127112,0.052282


Sweep outputs saved under: /content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/crlt/router_sweeps/pair_weight_weighted_logistic_v1


In [35]:
show_leaderboard(default_mode_drive_group_dir)


CR-LT leaderboard for pair_weight_weighted_logistic_v1


,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,mean_pred_support_fbeta,mean_best_minus_pred_support_fbeta,mean_pred_minus_baseline_support_fbeta,exact_accuracy,strategy_macro_f1,depth_accuracy,width_accuracy,elapsed_sec,status
0,weighted_logistic_pair1p0_point1p0_cost0p05_last1,weighted_logistic,NaN,1.0,1.0,0.05,0.317188,0.053822,0.041317,0.541667,0.261650,0.939951,0.541667,358.625983,completed
1,weighted_logistic_pair2p0_point1p0_cost0p05_last1,weighted_logistic,NaN,1.0,2.0,0.05,0.316716,0.054294,0.040845,0.620098,0.241537,0.920343,0.639706,337.766825,completed
2,weighted_logistic_pair0p5_point1p0_cost0p05_last1,weighted_logistic,NaN,1.0,0.5,0.05,0.310947,0.060063,0.035076,0.599265,0.233389,0.841912,0.618873,356.857891,completed


## Section 2: Alternative CR-LT Mode (`dynamic_margin`)

This section runs a simple sweep over `ALTERNATIVE_DYNAMIC_MARGIN_SCALES` while keeping the current best objective weights fixed at `pointwise_loss_weight = 1.0` and `pairwise_loss_weight = 2.0`.


In [36]:
ALTERNATIVE_MODE_SWEEP_GROUP_NAME = 'dynamic_margin_scale_sweep_v1'
ALTERNATIVE_DYNAMIC_MARGIN_SCALES = [0.25, 0.5, 1.0, 1.5]
ALTERNATIVE_POINTWISE_LOSS_WEIGHT = 1.0
ALTERNATIVE_PAIRWISE_LOSS_WEIGHT = 2.0
alternative_mode_configs = build_mode_configs(
    mode_name=ALTERNATIVE_PAIRWISE_LOSS_MODE,
    sweep_group_name=ALTERNATIVE_MODE_SWEEP_GROUP_NAME,
    pairwise_loss_weights=[ALTERNATIVE_PAIRWISE_LOSS_WEIGHT],
    reward_cost_weight=REWARD_COST_WEIGHT,
    pointwise_loss_weight=ALTERNATIVE_POINTWISE_LOSS_WEIGHT,
    dynamic_margin_scales=ALTERNATIVE_DYNAMIC_MARGIN_SCALES,
)
display(pd.DataFrame(alternative_mode_configs))


,sweep_group,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight
0,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,0.25,1.0,2.0,0.05
1,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,0.50,1.0,2.0,0.05
2,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,1.00,1.0,2.0,0.05
3,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,1.50,1.0,2.0,0.05


In [37]:
alternative_mode_drive_group_dir, alternative_mode_results = execute_sweep(alternative_mode_configs)



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_dynamic_margin_scale_sweep_v1_dynamic_margin_pair2p0_point1p0_cost0p05_mscale0p25_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pairwise-dyn

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.25,1.0,2.0,0.05,3,415.457887,...,0.191188,0.559558,0.022428,0.023116,0.01462,0.024662,0.017986,0.009519,0.083562,0.052282



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_dynamic_margin_scale_sweep_v1_dynamic_margin_pair2p0_point1p0_cost0p05_mscale0p5_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pairwise-dyna

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.50,1.0,2.0,0.05,3,356.712057,...,0.223036,0.283607,0.053606,0.021240,0.01462,0.024662,0.016928,0.010141,0.083562,0.052282
1,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.25,1.0,2.0,0.05,3,415.457887,...,0.191188,0.559558,0.022428,0.023116,0.01462,0.024662,0.017986,0.009519,0.083562,0.052282



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_dynamic_margin_scale_sweep_v1_dynamic_margin_pair2p0_point1p0_cost0p05_mscale1p0_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pairwise-dyna

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,1.00,1.0,2.0,0.05,3,301.865178,...,0.261728,0.119188,0.037039,0.019002,0.01462,0.024662,0.013752,0.008724,0.096826,0.052282
1,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.50,1.0,2.0,0.05,3,356.712057,...,0.223036,0.283607,0.053606,0.021240,0.01462,0.024662,0.016928,0.010141,0.083562,0.052282
2,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.25,1.0,2.0,0.05,3,415.457887,...,0.191188,0.559558,0.022428,0.023116,0.01462,0.024662,0.017986,0.009519,0.083562,0.052282



$ /usr/bin/python3 /content/project/Router_for_FeDeLiS/scripts/crlt/train_bert_crlt_support_ranker.py --query-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_query_table.jsonl --action-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_action_table.jsonl --pairwise-path /content/project/Router_for_FeDeLiS/preprocessed/crlt/router_pairwise_table.jsonl --output-dir /content/project/Router_for_FeDeLiS/outputs/crlt/bert_crlt_support_ranker_dynamic_margin_scale_sweep_v1_dynamic_margin_pair2p0_point1p0_cost0p05_mscale1p5_last1 --model-name distilbert-base-uncased --freeze-mode last1 --max-length 96 --pointwise-batch-size 32 --pairwise-batch-size 32 --epochs 12 --encoder-learning-rate 2e-05 --head-learning-rate 0.001 --weight-decay 0.01 --dropout 0.2 --n-splits 3 --early-stopping-patience 3 --seed 42 --pointwise-loss-weight 1.0 --pairwise-loss-weight 2.0 --reward-cost-weight 0.05 --pairwise-margin-weight 1.0 --pairwise-loss-mode dynamic_margin --pairwise-dyna

,sweep_group,run_name,status,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,n_splits,elapsed_sec,...,width_abs_error__std,cost_sensitive_error__std,loss__std,mean_pred_support_fbeta__std,mean_best_support_fbeta__std,mean_baseline_support_fbeta__std,mean_best_minus_pred_support_fbeta__std,mean_pred_minus_baseline_support_fbeta__std,mean_pred_llm_f1__std,mean_baseline_llm_f1__std
0,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,1.00,1.0,2.0,0.05,3,301.865178,...,0.261728,0.119188,0.037039,0.019002,0.01462,0.024662,0.013752,0.008724,0.096826,0.052282
1,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.50,1.0,2.0,0.05,3,356.712057,...,0.223036,0.283607,0.053606,0.021240,0.01462,0.024662,0.016928,0.010141,0.083562,0.052282
2,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,1.50,1.0,2.0,0.05,3,248.381251,...,0.334053,0.213626,0.146848,0.022506,0.01462,0.024662,0.016456,0.007885,0.097455,0.052282
3,dynamic_margin_scale_sweep_v1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,completed,dynamic_margin,0.25,1.0,2.0,0.05,3,415.457887,...,0.191188,0.559558,0.022428,0.023116,0.01462,0.024662,0.017986,0.009519,0.083562,0.052282


Sweep outputs saved under: /content/drive/MyDrive/MIT_nlp/project/Router_for_FeDeLiS/outputs/crlt/router_sweeps/dynamic_margin_scale_sweep_v1


In [38]:
show_leaderboard(alternative_mode_drive_group_dir)


CR-LT leaderboard for dynamic_margin_scale_sweep_v1


,run_name,pairwise_loss_mode,pairwise_dynamic_margin_scale,pointwise_loss_weight,pairwise_loss_weight,reward_cost_weight,mean_pred_support_fbeta,mean_best_minus_pred_support_fbeta,mean_pred_minus_baseline_support_fbeta,exact_accuracy,strategy_macro_f1,depth_accuracy,width_accuracy,elapsed_sec,status
0,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,1.00,1.0,2.0,0.05,0.328619,0.042391,0.052749,0.600490,0.290859,0.857843,0.600490,301.865178,completed
1,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,0.50,1.0,2.0,0.05,0.327923,0.043087,0.052053,0.522059,0.234308,0.682598,0.580882,356.712057,completed
2,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,1.50,1.0,2.0,0.05,0.326869,0.044142,0.050998,0.561275,0.264855,0.881127,0.580882,248.381251,completed
3,dynamic_margin_pair2p0_point1p0_cost0p05_mscal...,dynamic_margin,0.25,1.0,2.0,0.05,0.325849,0.045161,0.049979,0.522059,0.231420,0.704657,0.561275,415.457887,completed


## Section 3: CR-LT Objective Weight Sweep

This section sweeps the regression objective weight `pointwise_loss_weight` while keeping the pairwise mode fixed. This is useful when support-faithfulness gains appear sensitive to the balance between pointwise support regression and pairwise action ranking.


In [ ]:
OBJECTIVE_SWEEP_GROUP_NAME = 'objective_weight_crlt_v1'
OBJECTIVE_SWEEP_PAIRWISE_LOSS_MODE = DEFAULT_PAIRWISE_LOSS_MODE
OBJECTIVE_SWEEP_PAIRWISE_LOSS_WEIGHT = DEFAULT_PAIRWISE_LOSS_WEIGHT
OBJECTIVE_SWEEP_MARGIN_SCALE = DEFAULT_DYNAMIC_MARGIN_SCALE
objective_sweep_configs = build_objective_weight_configs(
    pairwise_loss_mode=OBJECTIVE_SWEEP_PAIRWISE_LOSS_MODE,
    sweep_group_name=OBJECTIVE_SWEEP_GROUP_NAME,
    pointwise_loss_weights=DEFAULT_POINTWISE_LOSS_WEIGHTS,
    reward_cost_weight=REWARD_COST_WEIGHT,
    pairwise_loss_weight=OBJECTIVE_SWEEP_PAIRWISE_LOSS_WEIGHT,
    dynamic_margin_scale=OBJECTIVE_SWEEP_MARGIN_SCALE,
)
display(pd.DataFrame(objective_sweep_configs))


In [ ]:
objective_drive_group_dir, objective_sweep_results = execute_sweep(objective_sweep_configs)


In [ ]:
show_leaderboard(objective_drive_group_dir)


## Notes

This notebook is CR-LT specific. It ranks models by support-faithfulness first, not by answer-only metrics. The dynamic-margin section is currently configured as a margin-scale sweep at fixed weights (`pointwise = 1.0`, `pairwise = 2.0`). Use 3-fold for sweep stages, then rerun only the best CR-LT setting with 5-fold and a longer epoch budget for the report result.
